# Retraining the SN and RB classifiers

This notebook reproduces the **training workflow** of two light-curve classifiers and demonstrates how to update them after a change in the data release or in the extracted features.

| Classifier | Positive class (`1`) | Labeled sample | Original DR24 settings |
| --- | --- | --- | --- |
| SN | Supernova | `data/sn_labels.csv` (`oid`, `is_sn`) | Random forest, `class_weight={0: 1, 1: 1000}` |
| RB | Real (non-artifact) | `data/rb_labels.csv` (`oid`, `label`) | Random forest, `n_estimators=831`, `max_depth=18` |

The curated **OID/label datasets are the reusable training samples**, independent of a specific feature extraction. The bundled DR24 examples additionally contain precomputed **ZTF r-band** features in the following format:

- `sid_{sn,rb}_dr24_snad_clf.dat`: `uint64` OIDs, in row order;
- `feature_{sn,rb}_dr24_snad_clf.dat`: matching row-major `float32` feature matrices;
- `feature_{sn,rb}_dr24_snad_clf.name`: ordered feature names (one per line);
- `feature_{sn,rb}_dr24_snad_clf.json`: JSON-serialized feature extractors.

The two DR24 extracts contain 47 features each, in identical order. Some labeled OIDs have no feature row in DR24: these are reported and excluded, **not relabeled or imputed**.

The notebook trains both classifiers using the provided data by default. For a new release or feature set, supply new feature files for the **same labeled OIDs** and adjust the paths in the configuration cell.

In [ ]:
# Optional: install runtime dependencies for a local training session.
# %pip install numpy pandas scikit-learn skl2onnx onnx onnxruntime
# %pip install pyarrow  # only for optional Parquet inputs

## 1. Configuration

Run from the repository root or from `docs/features`. DR24 binary feature files are the default inputs. For **DR23**, for example, change `FEATURE_RELEASE` and point `FEATURE_FILES` at the two new `feature_*.dat` paths. With `.dat` input, matching `sid_*.dat`, `feature_*.name`, and `feature_*.json` files are discovered automatically from the filename. Each task may use a different feature file; the number of features is read from `.name`, not hard-coded.

Feature input may alternatively be CSV/Parquet with an `oid` column and ordered numeric feature columns; in that case, set `EXTRACTOR_JSON_FILES` explicitly if you want the extractor configuration embedded in ONNX metadata.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate

DATA_DIR = next(
    (folder for folder in (Path("docs/features/data"), Path("data"))
     if (folder / "sn_labels.csv").is_file() and (folder / "rb_labels.csv").is_file()),
    Path("docs/features/data"),
)
FEATURE_RELEASE = "dr24"
BAND = "r"
RANDOM_SEED = 42
N_SPLITS = 5
OUTPUT_DIR = Path("models")

FEATURE_FILES = {
    task: DATA_DIR / f"feature_{task}_dr24_snad_clf.dat"
    for task in ("sn", "rb")
}
# Optional override; for .dat input, the adjacent .json is used by default.
EXTRACTOR_JSON_FILES = {"sn": None, "rb": None}

TASKS = {
    "sn": {
        "label": "is_sn",
        "positive_class": "supernova",
        "parameters": {"class_weight": {0: 1, 1: 1000}},
        "splitter": "stratified",
    },
    "rb": {
        "label": "label",
        "positive_class": "real (non-artifact) source",
        "parameters": {"n_estimators": 831, "max_depth": 18},
        "splitter": "kfold",
    },
}

# Used only if an external CSV/Parquet feature table contains auxiliary columns.
NON_FEATURE_COLUMNS = {
    "oid", "artefact", "is_sn", "regular_obj", "akb_ot", "label",
    "SN_clf_proba", "RB_clf_proba",
}
print("Data directory:", DATA_DIR.resolve())

## 2. Load labels and cross-match DR24 features by OID

The original extraction format guarantees that the *i*-th OID in `sid_*.dat` corresponds to the *i*-th feature row in `feature_*.dat`. **The order of the label CSV is independent of the order in the binary files**, so we explicitly cross-match by OID and preserve feature-column order from `.name`.

The loader checks duplicate OIDs, missing labels, incorrect binary sizes, missing/duplicate feature names, and non-finite values. It uses memory mapping for `.dat` and loads only labeled rows into memory.

In [ ]:
def read_labels(task):
    target = TASKS[task]["label"]
    labels = pd.read_csv(DATA_DIR / f"{task}_labels.csv",
                         usecols=["oid", target], dtype={"oid": "uint64"})
    if labels["oid"].isna().any() or labels["oid"].duplicated().any():
        raise ValueError(f"{task}: missing or duplicate labeled OIDs")
    if labels[target].isna().any() or not labels[target].isin([0, 1]).all():
        raise ValueError(f"{task}: labels must be 0 or 1")
    labels[target] = labels[target].astype(np.int64)
    return labels


def read_feature_table(task, labeled_oids):
    path = Path(FEATURE_FILES[task])
    if not path.is_file():
        raise FileNotFoundError(path)

    if path.suffix == ".dat":
        if not path.name.startswith("feature_"):
            raise ValueError("Expected binary filename feature_*.dat")
        sid_path = path.with_name("sid_" + path.name[len("feature_"):])
        name_path = path.with_suffix(".name")
        names = [name.strip() for name in name_path.read_text().splitlines()]
        if not names or any(not name for name in names) or len(set(names)) != len(names):
            raise ValueError(f"Invalid feature list: {name_path}")
        if sid_path.stat().st_size % np.dtype(np.uint64).itemsize:
            raise ValueError(f"Malformed OID file: {sid_path}")
        oids = np.memmap(sid_path, dtype=np.uint64, mode="r")
        if len(oids) == 0 or pd.Index(oids).has_duplicates:
            raise ValueError(f"Empty or duplicate OIDs: {sid_path}")
        expected = len(oids) * len(names) * np.dtype(np.float32).itemsize
        if path.stat().st_size != expected:
            raise ValueError(f"Expected {expected} feature bytes in {path}")

        positions = pd.Index(oids).get_indexer(labeled_oids)
        positions = positions[positions >= 0]
        array = np.memmap(path, dtype=np.float32, mode="r",
                          shape=(len(oids), len(names)))
        features = pd.DataFrame(np.asarray(array[positions]), columns=names)
        features.insert(0, "oid", np.asarray(oids[positions]))

    elif path.suffix == ".parquet" or path.suffix == ".csv" or path.name.endswith(".csv.gz"):
        frame = (pd.read_parquet(path) if path.suffix == ".parquet" else
                 pd.read_csv(path, dtype={"oid": "uint64"}))
        if "oid" not in frame.columns:
            raise ValueError(f"Missing oid column: {path}")
        names = [name for name in frame.columns if name not in NON_FEATURE_COLUMNS]
        features = frame[["oid", *names]]

    else:
        raise ValueError(f"Unsupported feature file: {path}")

    if not names or len(names) != len(set(names)) or "oid" in names:
        raise ValueError(f"{task}: missing or duplicate feature names")
    if features["oid"].isna().any() or features["oid"].duplicated().any():
        raise ValueError(f"{task}: missing or duplicate feature OIDs")
    return features, names


def make_dataset(task):
    labels = read_labels(task)
    features, names = read_feature_table(task, labels["oid"])
    label = TASKS[task]["label"]
    matched = labels.merge(features, on="oid", how="inner", validate="one_to_one", sort=False)
    n_missing = len(labels) - len(matched)
    X = matched[names].to_numpy(dtype=np.float32, copy=True)
    y = matched[label].to_numpy(dtype=np.int64)
    if not np.isfinite(X).all():
        raise ValueError(f"{task}: feature matrix contains NaN or Inf")
    counts = np.bincount(y, minlength=2)
    if min(counts) < N_SPLITS:
        raise ValueError(f"{task}: need at least {N_SPLITS} examples per class")
    print(f"{task.upper()}: {len(y):,}/{len(labels):,} matched OIDs, "
          f"{n_missing:,} missing, {len(names)} features, "
          f"class 0: {counts[0]:,}, class 1: {counts[1]:,}")
    return X, y, names


datasets = {task: make_dataset(task) for task in TASKS}

## 3. Cross-validation and final training

The random-forest settings are following: 5-fold **stratified CV** for SN and shuffled **KFold** for RB. Diagnostic scores are computed using cross-validation; **final exportable models are fitted to all matched labeled data**.

In [ ]:
def evaluate_and_train(task, X, y):
    settings = TASKS[task]
    model = RandomForestClassifier(random_state=RANDOM_SEED,
                                   **settings["parameters"])
    cv_type = StratifiedKFold if settings["splitter"] == "stratified" else KFold
    cv = cv_type(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
    scores = cross_validate(model, X, y, cv=cv,
                            scoring={"accuracy": "accuracy", "f1": "f1", "roc_auc": "roc_auc"},
                            n_jobs=1)
    results = {
        name: f"{scores['test_' + name].mean():.3f} ± "
              f"{scores['test_' + name].std():.3f}"
        for name in ("accuracy", "f1", "roc_auc")
    }
    model.fit(X, y)
    return model, results


models = {}
summary = []
for task, (X, y, feature_names) in datasets.items():
    model, metrics = evaluate_and_train(task, X, y)
    models[task] = model
    summary.append({"classifier": task.upper(), "objects": len(y),
                    "features": len(feature_names), **metrics})

pd.DataFrame(summary).set_index("classifier")

## 4. Export to ONNX and validate inference

Each model accepts `float_input: float32[N, n_features]`. The score is `P(class=1)` from `output_probability`: **supernova** for SN and **real/non-artifact** for RB.

To make the mapping between classical features and ONNX inputs explicit, ONNX metadata includes the **ordered** names from `.name`, the serialized extractor configuration from `.json`, the data release, and the class-1 meaning. The inference check below compares sklearn and ONNX Runtime probabilities, not the scientific validity of the training sample.

In [ ]:
def export_onnx(task, model, X, names):
    import onnxruntime as ort
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType

    source = Path(FEATURE_FILES[task])
    extractor_path = EXTRACTOR_JSON_FILES[task]
    if extractor_path is None and source.suffix == ".dat":
        extractor_path = source.with_suffix(".json")

    metadata = {
        "feature_names": json.dumps(names),
        "data_release": FEATURE_RELEASE,
        "band": BAND,
        "positive_class": "1",
        "positive_class_description": TASKS[task]["positive_class"],
        "training_sample": f"{task}_labels.csv",
        "n_training_objects": str(len(X)),
    }
    if extractor_path is not None:
        metadata["feature_extractor_json"] = json.dumps(
            json.loads(Path(extractor_path).read_text()), sort_keys=True
        )

    onx = convert_sklearn(
        model, initial_types=[("float_input", FloatTensorType([None, len(names)]))]
    )
    for key, value in metadata.items():
        prop = onx.metadata_props.add()
        prop.key, prop.value = key, value
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output = OUTPUT_DIR / f"{task.upper()}clf_{FEATURE_RELEASE}.onnx"
    output.write_bytes(onx.SerializeToString())

    session = ort.InferenceSession(str(output), providers=["CPUExecutionProvider"])
    sample = X[:min(32, len(X))].astype(np.float32, copy=False)
    input_name = session.get_inputs()[0].name
    probability_name = session.get_outputs()[1].name
    probabilities = session.run([probability_name], {input_name: sample})[0]
    if isinstance(probabilities, list):  # skl2onnx ZipMap output
        actual = np.array([row[1] for row in probabilities], dtype=np.float32)
    else:
        actual = np.asarray(probabilities)[:, 1]
    expected = model.predict_proba(sample)[:, list(model.classes_).index(1)]
    np.testing.assert_allclose(actual, expected, rtol=1e-4, atol=1e-5)
    print(f"Exported and checked: {output}")
    return output


exported = {
    task: export_onnx(task, models[task], X, names)
    for task, (X, y, names) in datasets.items()
}

## Retraining on another release or with new features

1. **Keep the same labeled samples** (`data/sn_labels.csv`, `data/rb_labels.csv`). Their OIDs and class labels are not derived from the feature files.
2. Fetch the curves or feature rows for these OIDs from your target release. If the extractor definition changes, **recompute** the features; editing a `.json` alone cannot update an existing `.dat` matrix.
3. Save the OIDs, features, ordered feature names, and extractor configuration as `sid_*.dat`, `feature_*.dat`, `feature_*.name`, and `feature_*.json`. Preserve row alignment between the first two binary files. Alternatively, supply a CSV or Parquet feature table with named columns and an `oid` column.
4. Set `FEATURE_RELEASE` and the two `FEATURE_FILES` paths above (and, for CSV/Parquet, optionally `EXTRACTOR_JSON_FILES`). Run all cells. The loaders cross-match **by OID**, report losses, and infer the new number/order of features automatically.
5. Review missing-object counts and CV metrics, and verify the exported ONNX models before releasing new scores. The positive-class semantics are fixed as `P(SN)` and `P(real)` respectively.

**Bundled DR24 cross-match:** 16,972/16,974 SN labels and 6,568/7,275 RB labels have DR24 features. Binary matrices and labels are committed separately on purpose: the curated samples remain reusable if features or the observing release change.